# Dataset Explorer — Pharmacovigilance Triage
Explore both datasets before building the environment.

**Datasets:**
- CADEC v2 — real patient forum posts with MedDRA annotations
- ADE Corpus v2 — medical literature sentences, binary labelled

In [1]:
import os
import re
import json
import random
import pandas as pd
from pathlib import Path
from collections import Counter

pd.set_option('display.max_colwidth', 300)
pd.set_option('display.max_rows', 50)

## Part 1 — CADEC v2

**Set `CADEC_PATH` to where you extracted the zip.**

In [2]:
# ---- CHANGE THIS to your local path ----
CADEC_PATH = "../data/raw/CADEC/data/CADEC.v2/cadec"

TEXT_DIR    = Path(CADEC_PATH) / "text"
ORIG_DIR    = Path(CADEC_PATH) / "original"
MEDDRA_DIR  = Path(CADEC_PATH) / "meddra"
SCT_DIR     = Path(CADEC_PATH) / "sct"

print("Text files found:", len(list(TEXT_DIR.glob('*.txt'))))

Text files found: 1250


In [6]:
def parse_ann(ann_path):
    """Parse a BRAT .ann file into a list of annotation dicts."""
    annotations = []
    if not ann_path.exists():
        return annotations
    with open(ann_path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) < 3:
                continue
            ann_id = parts[0]
            meta   = parts[1].split()
            text   = parts[2] if len(parts) > 2 else ''
            ann_type = meta[0]
            annotations.append({'id': ann_id, 'type': ann_type, 'text': text, 'meta': ' '.join(meta[1:])})
    return annotations


def load_cadec(n=None):
    """Load all CADEC records into a list of dicts."""
    records = []
    files = sorted(TEXT_DIR.glob('*.txt'))
    if n:
        files = files[:n]
    for txt_file in files:
        stem = txt_file.stem  # e.g. LIPITOR.30
        drug_name = stem.split('.')[0]
        report_id = stem

        with open(txt_file, encoding='utf-8') as f:
            report_text = f.read().strip()

        orig_anns   = parse_ann(ORIG_DIR   / f"{stem}.ann")
        meddra_anns = parse_ann(MEDDRA_DIR / f"{stem}.ann")
        sct_anns    = parse_ann(SCT_DIR    / f"{stem}.ann")

        adrs         = [a['text'] for a in orig_anns   if a['type'] == 'ADR']
        meddra_codes = [a['type'] for a in meddra_anns if not a['type'].startswith('CONCEPT')]
        has_unmapped = any('CONCEPT_LESS' in a['type'] for a in meddra_anns)

        records.append({
            'report_id':     report_id,
            'drug_name':     drug_name,
            'report_text':   report_text,
            'adr_count':     len(adrs),
            'adrs':          adrs,
            'meddra_codes':  meddra_codes,
            'has_unmapped':  has_unmapped,
            'text_length':   len(report_text),
        })
    return records


cadec = load_cadec()
df_cadec = pd.DataFrame(cadec)
print(f"Loaded {len(df_cadec)} CADEC records")
df_cadec

Loaded 1250 CADEC records


,report_id,drug_name,report_text,adr_count,adrs,meddra_codes,has_unmapped,text_length
0,ARTHROTEC.1,ARTHROTEC,"I feel a bit drowsy & have a little blurred vision, so far no gastric problems.\nI've been on Arthrotec 50 for over 10 years on and off, only taking it when I needed it.\nDue to my arthritis getting progressively worse, to the point where I am in tears with the agony, gp's started me on 75 twice...",4,"[bit drowsy, little blurred vision, gastric problems, feel a bit weird]","[10013649, 10005886, 10056819, 10025482]",False,483
1,ARTHROTEC.10,ARTHROTEC,"Hunger pangs.\nBrilliant, I have a new lease of life, i walk up & down steps properly, no longer sideways like a toddler, hip pain as gone other than if i jar it.",1,[Hunger pangs],[10033407],False,161
2,ARTHROTEC.100,ARTHROTEC,no side effects for the first two months .\nthen vaginal bleeding 2 wks after menstral cycle.\nstomach pain.\ncanker sores in my mouth.\nheadache.\nbeen off for 1 week still have bleeding.\nhelped my pain alot .\ntoo scared to take this drug again.,4,"[vaginal bleeding, stomach pain, canker sores, headache]","[10046883, 10042076, 10002959, 10019211]",False,241
3,ARTHROTEC.101,ARTHROTEC,"1st pill taken with food, a few hours after i experienced shortness of breath, a sense of depression, cramping, upset stomache will stop taking immediately.\nHonestly can not recommend this drug, I am still experiencing side effects just after 1 pill and im supposed to take twice/day 75mg.\nI ju...",4,"[shortness of breath, depression, cramping, upset stomache]","[10013968, 10012378, 10028294, 10046318]",False,340
4,ARTHROTEC.102,ARTHROTEC,"I have had no side effects been taking Arthrotec a little over a year, have not noticed any side effects.\nIt does help alot I noticed that when there are times when I forget to take it I can't stand or walk for any lengths of time.",0,[],[],False,231
...,...,...,...,...,...,...,...,...
1245,ZIPSOR.1,ZIPSOR,nausea.\nsome pain relief.,1,[nausea],[10028813],False,25
1246,ZIPSOR.2,ZIPSOR,Haven't really experienced any side effects thankfully.\nThis medicine seems to work very well.\nMy throat still hurts while on it; however the pain is tolerable and I am able to function through the day until my virus works its course and I am finally over this horrible sore throat!.,1,[hurts throat],[10033494],False,283
1247,ZIPSOR.3,ZIPSOR,"stiff neck, tightness in shoulders, muscle pain.\nnot recommended.",3,"[stiff neck, tightness in shoulders, muscle pain]","[10042043, 10028411]",True,65
1248,ZIPSOR.4,ZIPSOR,"Gave pretty good pain relief, with no side effects, and is very fast acting.\nBetter than mobic.",0,[],[],False,95


In [23]:
df_cadec.columns

Index(['report_id', 'drug_name', 'report_text', 'adr_count', 'adrs',
       'meddra_codes', 'has_unmapped', 'text_length'],
      dtype='str')

In [7]:
# Drug breakdown
print("Drug distribution")
print(df_cadec['drug_name'].value_counts().to_string())
print()
print("ADR count stats")
print(df_cadec['adr_count'].describe())
print()
print("Posts with unmapped MedDRA codes (hard cases)")
print(df_cadec['has_unmapped'].value_counts())

Drug distribution
drug_name
LIPITOR                 1000
ARTHROTEC                145
VOLTAREN                  46
VOLTAREN-XR               22
CATAFLAM                  10
DICLOFENAC-SODIUM          7
ZIPSOR                     5
CAMBIA                     4
PENNSAID                   4
DICLOFENAC-POTASSIUM       3
SOLARAZE                   3
FLECTOR                    1

ADR count stats
count    1250.000000
mean        5.054400
std         4.280993
min         0.000000
25%         2.000000
50%         4.000000
75%         7.000000
max        35.000000
Name: adr_count, dtype: float64

Posts with unmapped MedDRA codes (hard cases)
has_unmapped
False    1002
True      248
Name: count, dtype: int64


In [8]:
# View a single CADEC record in full detail
def show_cadec_record(report_id):
    row = df_cadec[df_cadec['report_id'] == report_id].iloc[0]
    print(f"{'='*60}")
    print(f"Report ID : {row['report_id']}")
    print(f"Drug      : {row['drug_name']}")
    print(f"Text      :\n{row['report_text']}")
    print(f"\nADRs ({row['adr_count']})  : {row['adrs']}")
    print(f"MedDRA codes : {row['meddra_codes']}")
    print(f"Has unmapped : {row['has_unmapped']}")

show_cadec_record('LIPITOR.73')

Report ID : LIPITOR.73
Drug      : LIPITOR
Text      :
A friend om mine took Lipitor (all statins have similar side effects).
He suffered terrible pains in hips, legs, hands etc Lost his muscular strength, had difficulties to swallow, problems with tinnitus and memory.
The problems appeared little by little over the years.
The best thing you can do for yourself and others is to educate yourself on this - and other medical issues.
Read books, read patient stories and look at The Lipitor Paradox at u-tube.
It`s a very funny and sad story.
Also visit spacedoc.net for more information about side effects.

ADRs (7)  : ['pains in legs', 'pains in hands', 'pains in hips', 'Lost muscular strength', 'difficulties to swallow', 'problems with memory', 'problems with tinnitus']
MedDRA codes : ['10033473', '10033430', '10033432', '10028350', '10042645', '10027175', '10043882']
Has unmapped : False


In [9]:
# Browse 10 random records
sample = df_cadec.sample(10, random_state=42)[['report_id','drug_name','report_text','adr_count','has_unmapped']]
sample

,report_id,drug_name,report_text,adr_count,has_unmapped
680,LIPITOR.558,LIPITOR,After 1 1/2 years of taking 10 mg/ day I am experiencing constant gas and diarrhea.\nHave stopped taking drug one week ago and hope this goes away.\nExcellent job of lowering my cholesterol!.,2,False
1102,LIPITOR.938,LIPITOR,cut high cholestorel from 278 to 134.,0,False
394,LIPITOR.30,LIPITOR,MUSCLE PAIN IN LEFT ARM WEAK FEELING GENERAL MALAISE.\nI WOULD NOT RECOMMENDTHIS DRUG.,3,False
930,LIPITOR.783,LIPITOR,no.,0,False
497,LIPITOR.393,LIPITOR,I have been on lipitor for 10 years for heart maintenance 20Mg.\nIwas changed to 10Mg and added tricor 145 Mg because of high ldl.\nI do experience leg weakness and cramping.\ncramping relieved with Potassium.,3,False
462,LIPITOR.361,LIPITOR,"Aching over my entire body, extreme fatigue, stiffness when getting up in the morning or getting up after being off my feet for awhile.\nDoctors make me feel like I am crazy.\nAlso had severe depression.\nHave taken 2-3 different statins.\nLike others who have written in, I have felt as though I...",4,False
950,LIPITOR.800,LIPITOR,"Fuzzy, dizzy, light-headedness feeling I've never had before.\nLow grade headache goes with it.\nMild insomnia.\nPoor concentration, noise intolerance.\nI rarely get headaches.\nStarted getting them a while back, never put 2 and 2 together.\nSaw an internist and 2 neurolgists, had MRI, MRA, all ...",7,True
81,ARTHROTEC.41,ARTHROTEC,"Dizziness, sickness, stomach gas, feel like a bubble of air is stuck in my chest.\nHadn't realised it was related until I read this but I get the hunger pangs too, I'm assuming this is the acids and gas in your stomach constantly churning over that make you feel like you're hungry.\nI've taken c...",13,True
43,ARTHROTEC.138,ARTHROTEC,"The only side effect that I experienced was the hungry feeling in my stomach ;however I have no complaints, just praise !\nDeborah McCready.",1,False
128,ARTHROTEC.84,ARTHROTEC,"No side affects.\nThis pill works like magic to releive my arthritic pain!\nI don't take it regularly, only when required - approximately once per month.",0,False


In [26]:
df_cadec['meddra_codes'].head()

0    [10013649, 10005886, 10056819, 10025482]
1                                  [10033407]
2    [10046883, 10042076, 10002959, 10019211]
3    [10013968, 10012378, 10028294, 10046318]
4                                          []
Name: meddra_codes, dtype: object

In [10]:
# Most common MedDRA codes
all_codes = [code for codes in df_cadec['meddra_codes'] for code in codes]
print("Top 20 MedDRA codes:")
for code, count in Counter(all_codes).most_common(20):
    print(f"{code}  →  {count} occurrences")

Top 20 MedDRA codes:
  10033371  →  484 occurrences
  10028411  →  278 occurrences
  10016256  →  136 occurrences
  10028294  →  129 occurrences
  10003239  →  127 occurrences
  10012378  →  124 occurrences
  10016766  →  121 occurrences
  10019211  →  111 occurrences
  10003549  →  105 occurrences
  10033473  →  79 occurrences
  10003993  →  76 occurrences
  10012735  →  75 occurrences
  10028350  →  74 occurrences
  10011301  →  73 occurrences
  10035067  →  72 occurrences
  10001949  →  71 occurrences
  10022437  →  66 occurrences
  10013573  →  66 occurrences
  10043890  →  64 occurrences
  10047810  →  62 occurrences


In [11]:
# Posts with the most ADRs — useful for Task 2 (complex cases)
df_cadec.sort_values('adr_count', ascending=False)[['report_id','drug_name','adr_count','adrs']].head(10)

,report_id,drug_name,adr_count,adrs
359,LIPITOR.269,LIPITOR,35,"[deppression, weakness, fatigue, muscle pain, stiffness, numbness, tingling, gas, bloating, indegestion, stuffy nose, plugged ears, ,lack of balance and cooridination, cough, bleeding in my abdomen after coughing and rupturing a blood vessel, decreased immune system, elevated wbc, trouble swallo..."
667,LIPITOR.546,LIPITOR,31,"[muscle pain, leg cramps, pain, Insomina, abnoraml dreams, depression, memory loss, photosensitivity, muscle pain, leg cramps, rhinitis, bronchitis, asthma, dry skin, tinnitus (left ear), taste loss, dry eyes, weight gain, hypertension, tinnitus, neck hurt, hips hurt, pain, Insomia, gained a lot..."
388,LIPITOR.295,LIPITOR,29,"[Terrible short term memory problems, lack of motivation, weak and wobbly unbalanced, walked like a 90 year old, not sure footed as I walked, overall weak feeling, muscle function irregularities, tightness in throat with occasional choking problem, lightheaded foggy brain (like being in a stupor..."
289,LIPITOR.205,LIPITOR,24,"[Alopecia, nausea, vomiting, severe back and hip pain, depression, exhaustion, muscle fatigue, sleeplessness, facial paralysis, alopecia, Nausea, vomiting, facial paralysis, hip and back pain, depression, paranoia, exhaustion, nausea, hair loss, hair loss, dark thoughts, thoughts of suicide, fac..."
715,LIPITOR.59,LIPITOR,23,"[Severe pain, cramping, Muscle weakness, atrophy, Weak & wobbley knees, Weak & wobbley wrists, Weak & wobbley ankles, Cramping in wrists & hands, Rib cage tightness, Constant fatigue, Insomnia, Mental fogginess, forgetfulness, Pancreatitis, Elevated liver enzymes, Chest tightness, breathlessness..."
82,ARTHROTEC.42,ARTHROTEC,22,"[nausea, dizziness, nausea, Stomach pains, chest pains, anxiety, fatigue, headache, erratic heart beat, raised blood pressure, dizziness, blurred vision, period like cramps, cramps, chest pains, elevated heart rate, hot flashes, dizziness, sever stomach cramps, weakness in my legs, thought I was..."
347,LIPITOR.258,LIPITOR,20,"[muscle weakness, memory loss, nerve damage, red spots all over my body, hair loss, slowed speach, shufflihg gait, unable to get out of chairs without help, ringing in my ears, sinus problems, pain, hairloss, spots on my body, weakness, eyes get blury, aphasia, brain not sending messages to my m..."
364,LIPITOR.273,LIPITOR,20,"[joint pain, extreme fatigue, tingling in feet, joint pains, pain, pain muscle, muscle stiffness, tingling in hands, legs tingling in, arms tingling in, inability to walk, contact pain where weight presses against things, muscle pains, couldn't walk, couldn't lift my arms, muscle pain, neck pain..."
291,LIPITOR.207,LIPITOR,19,"[charley horses in calves, charley horses in abdomen, charley horses in breasts, charley horses in shoulders, charley horses in sides, charley horses in feet, charley horses in feet, charley horses in legs, charley horses in abdomen, charley horses in breasts, charley horses in shoulders, charle..."
424,LIPITOR.327,LIPITOR,19,"[DIZZY, BALANCE, HIGHER BLOOD SUGARS, CARPAL TUNNAL, CRAMPS, JOINT PAIN, MEMORY LOSS, KIDNEY PROBLEMS, BALANCE PROBLEMS, BALANCE PROBLEMS, JOINT PAIN, KIDNEY READINGS BECAME BAD, TYPE II DIABETES BECAME UNCONTROLLED, JOINT PAIN, NEUROPATHY, PAIN IN MY HANDS, COULD NOT WALK WITHOUT FALLING, DEBIL..."


In [12]:
# Task 3 preview — Lipitor cluster (signal detection)
lipitor = df_cadec[df_cadec['drug_name'] == 'LIPITOR'].reset_index(drop=True)
print(f"Lipitor posts available for Task 3 clustering: {len(lipitor)}")
lipitor[['report_id','report_text','adr_count']].head(10)

Lipitor posts available for Task 3 clustering: 1000


,report_id,report_text,adr_count
0,LIPITOR.1,"Muscle pain, leg cramps, fatigue, irritable, pain in feet, neuropathy.\nI have stopped this medication a couple of times and my doctor keeps insisting I take it, rather then alternative medications.\nI'm getting off of this stuff after reading so many people are having similiar side-effects.\nDo...",6
1,LIPITOR.10,"In last 6 month, calf of left leg began to burn Nothing will stop it.\nNow, severe cramping.\nAlso have tinnitis, headaches, cannot get to sleep on my own,\nJust had surgery on my knee for torn maniscus, waiting to have shoulder surgery on rotator cuff, arthritis.\nI do not know if its the LIPIT...",5
2,LIPITOR.100,Severe skin rash to my upper thighs .\nThe itch and pain is unbearable to the point you cant sleep or bear to wear your trousers.,4
3,LIPITOR.1000,"Shoulder, knee, elbow, joint pain.\nMemory fog.\nAnxiety.\nLow Libido.\nAtorvastatin generic.\nExperienced brain fog immediatley.\nReally bad for my highly technical attention to detail work.\nStarted experiencing joint pain approx 6 months in.\nLeft shoulder is so bad right now have ice pack as...",11
4,LIPITOR.101,MOVE SLOW EYE HEAVY .NO SLEEP AT NIGHT AND DAY TIME NO SLEEP IN FOUR WEEK !.,4
5,LIPITOR.102,"Paranoia, anxiety.\nIt sure reduced my LDL cholesterol as well as triglycerides - but the psychiatric side effects were harsh.",3
6,LIPITOR.103,"Muscle aches.\nnow have fibromyalgia.\nMemory, learning problems, brain fog, depression, extreme fatigue.\nIn horrible pain, I quit taking it and replaced it with natural therapies.\nStarting to feel better.\nI hope and pray the damage isn't permanent.\nThis medication is nothing but a pharmaceu...",8
7,LIPITOR.104,"Terrible burning/numb sensation in hands & feet.\nJoint pain is worse than pre surgery!\nMy chiropractor advised me to look at this website.\nWill be quitting this.\nSince surgery, I've suddenly gone on blood pressure, cholesterol & diabetes meds.\nNever had these problems before.",3
8,LIPITOR.105,"Numbness in feet, leg, hip, neck cramps, eyes running, itchy and watery all the time like I was crying.\nDepression, itching all the time.\nMakes you feel 25 years older.\nI quit taking it most problems gone in 3 weeks.\nI will try to control with diet and exercise, and omega 3 fish oil, will ne...",8
9,LIPITOR.106,"Rough anxiety and paranoia.\nTo be fair, I must note that I was also taking Norvasc - a blood pressure pill and when the above symptoms persisted I stopped Norvasc at once (replaced with Diovan).\nHowever, at that time - over a month ago - I was only aware of Norvasc's side effects and, indeed, ...",4


---
## Part 2 — ADE Corpus v2 (from Hugging Face)

If HF download fails on your network, load from local Parquet instead.

In [13]:
try:
    from datasets import load_dataset
    ds = load_dataset('ade_corpus_v2', 'Ade_corpus_v2_classification')
    df_ade = ds['train'].to_pandas()
    print("Loaded from Hugging Face")
except Exception as e:
    print(f"HF download failed: {e}")
    print("Tip: Download manually from https://huggingface.co/datasets/ade_corpus_v2")
    print("Click Files → train-00000-of-00001.parquet → Download")
    print("Then run: df_ade = pd.read_parquet('path/to/train-00000-of-00001.parquet')")
    df_ade = None

if df_ade is not None:
    print(f"Total rows: {len(df_ade)}")

D:\Hackathon\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 23516/23516 [00:00<00:00, 1337962.44 examples/s]

Loaded from Hugging Face
Total rows: 23516


In [22]:
df_ade

,text,label,text_length
0,Intravenous azithromycin-induced ototoxicity.,1,45
1,"Immobilization, while Paget's bone disease was present, and perhaps enhanced activation of dihydrotachysterol by rifampicin, could have led to increased calcium-release into the circulation.",1,190
2,Unaccountable severe hypercalcemia in a patient treated for hypoparathyroidism with dihydrotachysterol.,1,103
3,METHODS: We report two cases of pseudoporphyria caused by naproxen and oxaprozin.,1,81
4,METHODS: We report two cases of pseudoporphyria caused by naproxen and oxaprozin.,1,81
...,...,...,...
23511,"At autopsy, the liver was found to be small, shrunken, and scarred; histological sections demonstrated postnecrotic cirrhosis.",0,126
23512,"Physical exam revealed a patient with aphasia, tremor, and an expressionless face, able to make eye contact and move all four extremities.",0,138
23513,"At the time when the leukemia appeared seven of the patients were in complete, and one in partial, remission as regards the ovarian carcinoma.",0,142
23514,The American Society for Regional Anesthesia and Pain Medicine has specific guidelines for treatment of these patients when they undergo neuraxial anesthetic procedures.,0,169


In [14]:
if df_ade is not None:
    print("Label distribution")
    print(df_ade['label'].value_counts())
    print()
    print("Label names")
    print("0 = Not ADE (non_serious candidate)")
    print("1 = ADE Related (serious candidate)")

Label distribution
label
0    16695
1     6821
Name: count, dtype: int64

Label names
0 = Not ADE (non_serious candidate)
1 = ADE Related (serious candidate)


In [15]:
if df_ade is not None:
    print("10 ADE examples (label=1)")
    df_ade[df_ade['label'] == 1].sample(10, random_state=42)[['text', 'label']]

10 ADE examples (label=1)


In [16]:
if df_ade is not None:
    print("10 non-ADE examples (label=0)")
    df_ade[df_ade['label'] == 0].sample(10, random_state=42)[['text', 'label']]

10 non-ADE examples (label=0)


In [17]:
if df_ade is not None:
    print("Text length distribution")
    df_ade['text_length'] = df_ade['text'].str.len()
    print(df_ade.groupby('label')['text_length'].describe())

Text length distribution
         count        mean        std   min    25%    50%    75%    max
label                                                                  
0      16695.0  124.703983  57.901832   7.0   84.0  116.0  156.0  742.0
1       6821.0  152.407858  70.045387  17.0  100.0  144.0  193.0  568.0


---
## Part 3 — Cross-dataset comparison

Quick sanity check on how the two datasets differ.

In [18]:
print("CADEC v2")
print(f"  Records          : {len(df_cadec)}")
print(f"  Avg text length  : {df_cadec['text_length'].mean():.0f} chars")
print(f"  Avg ADRs/report  : {df_cadec['adr_count'].mean():.1f}")
print(f"  Hard cases (CONCEPT_LESS): {df_cadec['has_unmapped'].sum()}")
print(f"  Text style       : Informal patient forum posts")
print()
if df_ade is not None:
    print("ADE Corpus v2 ")
    print(f"  Records          : {len(df_ade)}")
    print(f"  ADE (label=1)    : {(df_ade['label']==1).sum()}")
    print(f"  Non-ADE (label=0): {(df_ade['label']==0).sum()}")
    print(f"  Avg text length  : {df_ade['text_length'].mean():.0f} chars")
    print(f"  Text style       : Formal medical literature sentences")

CADEC v2
  Records          : 1250
  Avg text length  : 459 chars
  Avg ADRs/report  : 5.1
  Hard cases (CONCEPT_LESS): 248
  Text style       : Informal patient forum posts

ADE Corpus v2 
  Records          : 23516
  ADE (label=1)    : 6821
  Non-ADE (label=0): 16695
  Avg text length  : 133 chars
  Text style       : Formal medical literature sentences


In [19]:
print(" Task mapping ")
print()
print("Task 1 (Easy)   → ADE Corpus v2, binary label, clean text")
print("Task 2 (Medium) → CADEC, multi-label, noisy patient text, confounders")
print("Task 3 (Hard)   → CADEC Lipitor cluster, batch of 5, signal detection")

 Task mapping 

Task 1 (Easy)   → ADE Corpus v2, binary label, clean text
Task 2 (Medium) → CADEC, multi-label, noisy patient text, confounders
Task 3 (Hard)   → CADEC Lipitor cluster, batch of 5, signal detection
